# MTAH 2027 — possession Barça–Athletic

Pipeline final : tracking ATD (ballon) → *ballon mort* par modèle GBM (7 matchs Figshare) → trous du ballon comblés par les détections SAM3 (pré-calculées, lues depuis le dataset) + Viterbi/Kalman → équipe la plus proche + lissage par vote majoritaire.

In [ ]:
import os, sys, glob, subprocess, pickle, io, contextlib, math, warnings
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd, cv2, requests, joblib
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.ensemble import HistGradientBoostingClassifier

REPO = Path("/kaggle/working/mtah")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/metrica-sports/more-than-a-hack-2027", str(REPO)], check=True)
sys.path[:0] = [str(REPO), str(REPO / "evaluation")]

In [ ]:
import possession_metrics as pm                      # le scorer officiel
from games import GAMES, GAME, half_video_blocks

def find(pattern, required=True):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits or not required, f"introuvable : {pattern}"
    return hits[0] if hits else None

ATD_TXT, ATD_XML, VIDEO = find("*tactical_atd.txt"), find("*tactical_atd.xml"), find("*tactical.mp4")
HOM = pd.read_csv(find("*tactical_homography.csv"))
HOM_M = HOM[[f"h{r}{c}" for r in range(3) for c in range(3)]].values.astype(float).reshape(-1, 3, 3)

FPS, T = 25, GAMES[GAME]["n_video_frames"]
PL, PW = GAMES[GAME]["pitch"]
HALVES = [(a, b) for _, a, b in half_video_blocks(GAME)]   # bornes incluses, en images vidéo
H, A, D, X = pm.H, pm.A, pm.D, pm.X                  # 0, 1, 2, 3

gt_codes = pm.label_codes(pm.read_label_csv(str(REPO / "evaluation/gt/BAR-ATH_possession_gt_public.csv")))

In [ ]:
# --- Équipes des pistes et ballon ATD (index = image vidéo = frame_id - 1) ---
root = ET.parse(ATD_XML).getroot()
team_ids = [t.get("id") for t in root.iter("Team")]
TEAM = np.array([0 if p.get("teamId") == team_ids[0] else 1 for p in root.iter("Player")], dtype=np.int8)  # 0 = H, 1 = A

def xyv(g):
    if g[0] == "N":
        return (np.nan, np.nan, np.nan)
    x, y, v = g.split(",")[:3]
    return (float(x), float(y), float(v))

K, NMIN, LAG = 46, 8, 10
B = np.full((T, 3), np.nan, dtype=np.float32)        # ballon : x, y, vitesse
PA = np.full((T, K, 3), np.nan)                      # joueurs en mètres : x, y, S (inutilisée)
with open(ATD_TXT) as f:
    for line in f:
        parts = line.rstrip("\n").split(":")
        if len(parts) != 3:
            continue
        t = int(parts[0]) - 1                        # frame_id démarre à 1
        if not 0 <= t < T:
            continue
        B[t] = xyv(parts[2])
        PA[t] = np.array(parts[1].replace(";", ",").split(","), dtype=float).reshape(K, 3)
B[:, 0] *= PL; B[:, 1] *= PW                         # 0-1 -> mètres
PA[..., 0] *= PL; PA[..., 1] *= PW
VIS = np.isfinite(B[:, 0])                           # ballon présent dans l'ATD
INSIDE = np.zeros(T, bool)
for lo, hi in HALVES:
    INSIDE[lo:min(hi, T - 1) + 1] = True
print(f"pistes : {len(TEAM)} | ballon visible : {VIS.mean():.1%}")

## 1. Modèle « ballon mort » (vitesses des joueurs, 7 matchs Figshare 28196177)

Modèle final à 16 features (10 de base + 6 extras), seuil de `duree_calme` global.

In [ ]:
ART_ID = 28196177
STEP, TAU, V_SPRINT = 5, 0.5, 5.5                    # sous-échantillonnage 5 Hz ; seuils immobilité / sprint (m/s)
W3, W5 = 3 * FPS, 5 * FPS
IDX = list(range(14, 21))                            # fichiers « positions_raw_observed » (7 matchs)
BASE10 = ["v_moy", "v_std", "v_q90", "f_immo", "f_sprint", "a_abs_moy", "dv_3", "f_immo_roll_3", "v_moy_min_5", "duree_calme"]
EXTRAS = ["dv_1", "dv_5", "f_immo_roll_1", "f_immo_roll_5", "v_moy_std_3", "a_abs_roll_3"]
FEATS = BASE10 + EXTRAS

FILES = requests.get(f"https://api.figshare.com/v2/articles/{ART_ID}", timeout=30).json()["files"]

def telecharger(i):
    f = FILES[i]
    with requests.get(f["download_url"], stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(f["name"], "wb") as out:
            for chunk in r.iter_content(1 << 20):
                out.write(chunk)
    return f["name"]

def lire_tracking(path):
    ball, joueurs = {}, {}
    for _, el in ET.iterparse(path, events=("end",)):
        if el.tag != "FrameSet":
            continue
        sec, team, pid = el.get("GameSection"), el.get("TeamId"), el.get("PersonId")
        frames = el.findall("Frame")
        if frames:
            N = np.array([int(f.get("N")) for f in frames], dtype=np.int64)
            if team == "BALL":
                s = pd.Series([f.get("BallStatus") for f in frames], index=N)
                s = s[~s.index.duplicated()].sort_index()
                ball[sec] = s[s.isin(["0", "1"])]
            else:
                joueurs.setdefault(sec, {})[pid] = dict(
                    N=N, X=np.array([float(f.get("X", "nan")) for f in frames]),
                    Y=np.array([float(f.get("Y", "nan")) for f in frames]),
                    S=np.array([float(f.get("S", "nan")) for f in frames]))
        el.clear()
    return ball, joueurs

def detecter_unite(joueurs):                         # S en m/s ou km/h ?
    ratios = []
    for sec in joueurs.values():
        for d in sec.values():
            dN = np.diff(d["N"])
            dist = np.hypot(np.diff(d["X"]), np.diff(d["Y"])) * FPS
            ok = (dN == 1) & (dist > 1.0) & np.isfinite(d["S"][1:])
            ratios.extend((d["S"][1:][ok] / dist[ok]).tolist())
    if not ratios:
        return 3.6
    return 3.6 if float(np.median(ratios)) > 2 else 1.0

def duree_consecutive(mask, fps=FPS):
    grp = (mask != mask.shift()).cumsum()
    return (mask.groupby(grp).cumcount() + 1) * mask / fps

def features_section(V):                             # V : images x joueurs, vitesses m/s
    arr = V.to_numpy(dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        n = np.isfinite(arr).sum(axis=1).astype(float)
        n[n == 0] = np.nan
        v_moy = np.nanmean(arr, axis=1)
        v_std = np.nanstd(arr, axis=1)
        v_q90 = np.nanquantile(arr, 0.9, axis=1)
        f_immo = (arr < TAU).sum(axis=1) / n
        f_sprint = (arr >= V_SPRINT).sum(axis=1) / n
        Ac = V.rolling(5, min_periods=1).mean().diff().to_numpy() * FPS
        a_abs_moy = np.nanmean(np.abs(Ac), axis=1)
    df = pd.DataFrame({"v_moy": v_moy, "v_std": v_std, "v_q90": v_q90, "f_immo": f_immo,
                       "f_sprint": f_sprint, "a_abs_moy": a_abs_moy}, index=V.index)
    df["dv_3"] = df["v_moy"] - df["v_moy"].shift(W3)
    df["f_immo_roll_3"] = df["f_immo"].rolling(W3, min_periods=1).mean()
    df["v_moy_min_5"] = df["v_moy"].rolling(W5, min_periods=1).min()
    return df

def ajouter_extras(F):
    F = F.copy()
    vm = F["v_moy"]
    F["dv_1"] = vm - vm.shift(FPS)
    F["dv_5"] = vm - vm.shift(W5)
    F["f_immo_roll_1"] = F["f_immo"].rolling(FPS, min_periods=1).mean()
    F["f_immo_roll_5"] = F["f_immo"].rolling(W5, min_periods=1).mean()
    F["v_moy_std_3"] = vm.rolling(W3, min_periods=2).std()
    F["a_abs_roll_3"] = F["a_abs_moy"].rolling(W3, min_periods=1).mean()
    return F

def base_match(path):                                # features à 25 Hz d'un match (sans duree_calme)
    ball, joueurs = lire_tracking(path)
    div = detecter_unite(joueurs)
    out = []
    for sec in sorted(ball, key=lambda s: (0 if "first" in s.lower() else 1, s)):
        idx = ball[sec].index
        V = pd.DataFrame({pid: pd.Series(d["S"], index=d["N"]).loc[lambda s: ~s.index.duplicated()]
                          for pid, d in joueurs.get(sec, {}).items()}).reindex(idx) / div
        df = features_section(V)
        df["y"] = (ball[sec] == "0").astype(int)     # 1 = ballon mort
        df["section"], df["N"] = sec, idx
        out.append(df)
    return pd.concat(out)

bases = []
for k, i in enumerate(IDX):
    cache = f"base_{i}.pkl"
    if os.path.exists(cache):
        b = pd.read_pickle(cache)
    else:
        b = base_match(telecharger(i)); os.remove(FILES[i]["name"]); b.to_pickle(cache)
    b["match"] = k
    bases.append(b)

pool = pd.concat(bases, ignore_index=True)
SEUIL_F = float(pool["v_moy"].median())              # seuil global de duree_calme
parts = []
for _, g in pool.groupby(["match", "section"], sort=False):
    g = g.copy()
    g["duree_calme"] = duree_consecutive(g["v_moy"] < SEUIL_F)
    parts.append(ajouter_extras(g))
pool = pd.concat(parts)
pool = pool[pool["N"] % STEP == 0].reset_index(drop=True)
GBM = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300,
                                     class_weight="balanced", random_state=0).fit(pool[FEATS], pool["y"])
print(f"seuil global {SEUIL_F:.2f} m/s | {len(pool)} lignes | modèle entraîné sur {pool['match'].nunique()} matchs")

In [ ]:
# --- Application au match ATD : vitesses causales sur 0,4 s, proba lissée sur 2 s, seuil 0,5, arrêt >= 2 s ---
def lag(a):
    out = np.full_like(a, np.nan); out[LAG:] = a[:-LAG]; return out
vc = np.hypot(PA[..., 0] - lag(PA[..., 0]), PA[..., 1] - lag(PA[..., 1])) / LAG * FPS

parts = []
for lo, hi in HALVES:
    hi = min(hi, T - 1)
    V = pd.DataFrame(vc[lo:hi + 1], index=np.arange(lo, hi + 1))
    Fh = features_section(V)
    Fh["duree_calme"] = duree_consecutive(Fh["v_moy"] < SEUIL_F)
    Fh = ajouter_extras(Fh)
    Fh["n"] = V.notna().sum(axis=1)
    parts.append(Fh)
Fe = pd.concat(parts).reindex(range(T))

ok = Fe["n"].ge(NMIN)
p = pd.Series(np.nan, index=Fe.index)
p[ok] = GBM.predict_proba(Fe.loc[ok, FEATS])[:, 1]
p_s = p.interpolate(limit=2 * FPS).rolling(2 * FPS + 1, center=True, min_periods=1).median().to_numpy()

def to_dead(ps, thr, min_s):
    d = np.diff(np.r_[0, ((ps >= thr) & INSIDE).astype(int), 0])
    out = np.zeros(T, bool)
    for s, e in zip(np.where(d == 1)[0], np.where(d == -1)[0]):
        if e - s >= min_s * FPS:
            out[s:e] = True
    return out

DEAD = to_dead(p_s, 0.5, 2.0)
gt_ok, gt_d = gt_codes != X, gt_codes == D
tp, fp, fn = (DEAD & gt_d & gt_ok).sum(), (DEAD & ~gt_d & gt_ok).sum(), (~DEAD & gt_d & gt_ok).sum()
print(f"ballon mort (modèle) : {DEAD[INSIDE].mean():.1%} du temps de jeu | D dans la vérité : {gt_d[gt_ok].mean():.1%} | "
      f"précision {tp/(tp+fp):.3f} rappel {tp/(tp+fn):.3f}")

## 2. Ballon manquant : détections SAM3 (pré-calculées) + Viterbi/Kalman

Les détections `soccer ball` (seuil 0,2, images pleines) ont été calculées une fois et sont lues depuis le dataset.

In [ ]:
SAM_PKL = find("sam3_candidats.pkl")
SAM_EXTRA = find("sam3_candidats_extra.pkl", required=False)

rows = pickle.load(open(SAM_PKL, "rb"))              # [(frame, boxes, scores), ...]
if SAM_EXTRA:
    rows += pickle.load(open(SAM_EXTRA, "rb"))
rows = sorted({r[0]: r for r in rows}.values(), key=lambda r: r[0])
SAM = pd.DataFrame([(f, len(s), max(s, default=np.nan), b, s) for f, b, s in rows],
                   columns=["frame", "n", "score", "boxes", "scores"]).set_index("frame")
SAM["dt"] = SAM.index.to_series().diff().fillna(0).astype(int)
SAM["segment"] = (SAM["dt"] != 1).cumsum()           # runs d'images consécutives

NEED = INSIDE & ~VIS & ~DEAD                         # ballon vivant, absent de l'ATD
print(f"{len(SAM)} images SAM3 | à combler : {NEED.sum()} ({NEED.sum() / INSIDE.sum():.1%} du jeu)")

In [ ]:
def project(M, x, y):
    den = M[2, 0] * x + M[2, 1] * y + M[2, 2]
    return (M[0, 0] * x + M[0, 1] * y + M[0, 2]) / den, (M[1, 0] * x + M[1, 1] * y + M[1, 2]) / den

def unproject(M, u, v):
    return project(np.linalg.inv(M), u, v)

MARGIN, MAX_PX, BALL_D = 2.0, 25, 0.22           # m au-delà des lignes, taille max d'une boîte (px), diamètre du ballon (m)
MU, SIG_SZ = np.log(1.6), 0.4                    # log(taille boîte / taille attendue) : centre, écart-type
ABS, SIG, VMAX = np.log(0.25), 0.35, 35 / FPS    # log-vraisemblance « absent », écart-type du pas (m/image), vitesse max (m/image)
COL_W = 4.0                                      # pénalité boîte rouge/orange/jaune saturée (corners, maillots)
Q_VAR, R_VAR = 0.1, 0.25                         # Kalman : variance d'accélération, variance de mesure (m²)

def expected_px(f, u, v):                        # taille attendue du ballon (px), via l'homographie
    Mx = HOM_M[f]
    p0 = np.array(project(Mx, u, v))
    return max(np.linalg.norm(np.array(project(Mx, u + BALL_D / PL, v)) - p0),
               np.linalg.norm(np.array(project(Mx, u, v + BALL_D / PW)) - p0))

def smooth(z):                                   # lisseur RTS, vitesse constante par axe ; z (n,2) en m, NaN = absent
    n = len(z)
    F_ = np.array([[1., 1.], [0., 1.]]); Q_ = Q_VAR * np.array([[.25, .5], [.5, 1.]])
    st = np.full((n, 2, 2), np.nan)
    for ax in (0, 1):
        y = z[:, ax]; okk = np.isfinite(y)
        if not okk.any():
            continue
        i0 = int(np.argmax(okk))
        xf, xp = np.zeros((n, 2)), np.zeros((n, 2))
        Pf, Pp = np.zeros((n, 2, 2)), np.zeros((n, 2, 2))
        x, Pc = np.array([y[i0], 0.]), np.diag([R_VAR, 1.0])
        for t_ in range(i0, n):
            if t_ > i0:
                x, Pc = F_ @ x, F_ @ Pc @ F_.T + Q_
            xp[t_], Pp[t_] = x, Pc
            if okk[t_]:
                kg = Pc[:, 0] / (Pc[0, 0] + R_VAR)
                x, Pc = x + kg * (y[t_] - x[0]), Pc - np.outer(kg, Pc[0])
            xf[t_], Pf[t_] = x, Pc
        xs_ = xf.copy()
        for t_ in range(n - 2, i0 - 1, -1):
            xs_[t_] = xf[t_] + Pf[t_] @ F_.T @ np.linalg.inv(Pp[t_ + 1]) @ (xs_[t_ + 1] - xp[t_ + 1])
        st[i0:, ax] = xs_[i0:]
    return st[:, :, 0], np.hypot(st[:, 0, 1], st[:, 1, 1]) * FPS

In [ ]:
def warm(img, b):                                # fraction of saturated red/orange/yellow pixels in a box
    crop = img[max(int(b[1]), 0):int(np.ceil(b[3])), max(int(b[0]), 0):int(np.ceil(b[2]))]
    if crop.size == 0:
        return 0.0
    h, s, v = cv2.split(cv2.cvtColor(crop, cv2.COLOR_BGR2HSV))
    return float((((h <= 35) | (h >= 165)) & (s > 100) & (v > 100)).mean())

def cands(f, boxes, scores, img):                # valid candidates of one frame: positions (m) and log-emissions
    pos, emi = [], []
    for b, s in zip(boxes, scores):
        w = max(b[2] - b[0], b[3] - b[1])
        u, v = unproject(HOM_M[f], (b[0] + b[2]) / 2, (b[1] + b[3]) / 2)
        x, y = u * PL, v * PW
        if w > MAX_PX or not (-MARGIN <= x <= PL + MARGIN and -MARGIN <= y <= PW + MARGIN):
            continue
        z = (np.log(w / expected_px(f, u, v)) - MU) / SIG_SZ
        pos.append((x, y)); emi.append(np.log(s) - 0.5 * z * z - COL_W * warm(img, b))
    return np.array(pos).reshape(-1, 2), np.array(emi)

def decode(pos, emi):                            # Viterbi; states per frame: K candidates + "absent" (remembers last position)
    S, A, M = [emi[0]], [ABS], [(None, 0)]       # candidate scores, absent score, absent memory (last position, age)
    bc, ba = [], []                              # backpointers: -1 = comes from "absent", j = from candidate j
    for t in range(1, len(pos)):
        a, b = pos[t - 1], pos[t]
        last, age = M[t - 1]
        from_a = np.full(len(b), A[t - 1])       # absent -> candidate: distance to the last known position
        if last is not None and len(b):
            d = np.linalg.norm(b - last, axis=1)
            from_a = A[t - 1] - 0.5 * (d / (age + 1) / SIG) ** 2
            from_a[d > VMAX * (age + 1)] = -np.inf
        best_c, arg_c = np.full(len(b), -np.inf), np.zeros(len(b), int)
        if len(a) and len(b):                    # candidate -> candidate: one frame, speed capped
            d = np.linalg.norm(a[:, None] - b[None], axis=2)
            c = S[t - 1][:, None] - 0.5 * (d / SIG) ** 2
            c[d > VMAX] = -np.inf
            best_c, arg_c = c.max(0), c.argmax(0)
        use_c = best_c > from_a
        S.append(np.where(use_c, best_c, from_a) + emi[t])
        bc.append(np.where(use_c, arg_c, -1))
        sc_a = A[t - 1] + ABS                    # -> absent: from the previous absent, or from the best candidate
        j = int(np.argmax(S[t - 1])) if len(a) else -1
        if j >= 0 and S[t - 1][j] + ABS > sc_a:
            A.append(S[t - 1][j] + ABS); ba.append(j); M.append((a[j], 1))
        else:
            A.append(sc_a); ba.append(-1); M.append((last, age + 1))
    k = int(np.argmax(S[-1])) if len(S[-1]) and S[-1].max() > A[-1] else -1
    path = [k]
    for t in range(len(pos) - 1, 0, -1):
        k = int(bc[t - 1][k]) if k >= 0 else int(ba[t - 1])
        path.append(k)
    return path[::-1]                            # chosen candidate per frame, -1 = absent

def track(seg):                                  # seg: rows of df with consecutive frames
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(seg.index[0]))
    pe = [cands(f, b, s, cap.read()[1]) for f, b, s in zip(seg.index, seg["boxes"], seg["scores"])]
    cap.release()
    path = decode([p for p, _ in pe], [e for _, e in pe])
    z = np.array([p[k] if k >= 0 else (np.nan, np.nan) for (p, _), k in zip(pe, path)])
    xy, sp = smooth(z)
    det = np.isfinite(z[:, 0])
    out = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "speed_ms": sp, "det": det}, index=seg.index)
    if det.any():                                # no extrapolation before the first / after the last detection
        i0, i1 = np.flatnonzero(det)[[0, -1]]
        out.iloc[:i0, :3] = np.nan; out.iloc[i1 + 1:, :3] = np.nan
    return out

In [ ]:
seg_ids = SAM.loc[SAM.index.isin(np.flatnonzero(NEED)), "segment"].unique()
sub = SAM[SAM["segment"].isin(seg_ids)]
SAMTRK = pd.concat([track(seg) for _, seg in tqdm(sub.groupby("segment"), total=len(seg_ids), desc="Viterbi")])
SAMTRK.to_pickle("sam3_viterbi_track.pkl")
print(f"{len(seg_ids)} segments | {len(SAMTRK)} images | positions finies : {np.isfinite(SAMTRK['x']).mean():.1%} "
      f"| détectées : {SAMTRK['det'].mean():.1%}")

## 3. Possession : équipe la plus proche du ballon + vote majoritaire ±w

In [ ]:
def majority(codes, w):
    if w <= 0:
        return codes.copy()
    ch = np.concatenate(([0], np.cumsum(codes == H)))
    ca = np.concatenate(([0], np.cumsum(codes == A)))
    i = np.arange(len(codes))
    lo, hi = np.maximum(0, i - w), np.minimum(len(codes), i + w + 1)
    h, a = ch[hi] - ch[lo], ca[hi] - ca[lo]
    out = np.full(len(codes), D, dtype=np.int8)
    seen = (h + a) > 0
    out[seen] = np.where(h[seen] >= a[seen], H, A)
    return out

In [ ]:
BALL = np.full((T, 2), np.nan)
SRC = np.zeros(T, np.int8)                            # 0 = inconnu, 1 = ATD, 2 = SAM3 + Viterbi
BALL[VIS] = B[VIS, :2]; SRC[VIS] = 1

fi = SAMTRK.index.to_numpy()
use = NEED[fi] & np.isfinite(SAMTRK["x"].to_numpy())
BALL[fi[use]] = SAMTRK[["x", "y"]].to_numpy()[use]; SRC[fi[use]] = 2

def nearest(ball):                                    # équipe du joueur le plus proche, D si ballon inconnu
    d = np.hypot(PA[..., 0] - ball[:, None, 0], PA[..., 1] - ball[:, None, 1])
    d = np.where(np.isfinite(d), d, np.inf)
    idx, dist = d.argmin(1), d.min(1)
    out = np.full(T, D, np.int8)
    okk = np.isfinite(dist)
    out[okk] = TEAM[idx[okk]]
    return out

near_atd = nearest(np.where(VIS[:, None], B[:, :2], np.nan))
near_full = nearest(BALL)

alive = INSIDE & ~DEAD
print(f"ballon vivant ({alive.sum()} images) : ATD {(alive & (SRC == 1)).sum() / alive.sum():.1%} | "
      f"SAM3 {(alive & (SRC == 2)).sum() / alive.sum():.1%} | inconnu (-> D) {(alive & (SRC == 0)).sum() / alive.sum():.1%}")

In [ ]:
def build(near, w, dead=None):
    out = majority(near, w)
    if dead is not None:
        out[dead] = D
    return out

rows = []
for w in (0, 25, 50, 75, 100, 125, 150):
    for name, near, dead in (("1 ATD seul", near_atd, None),
                             ("2 + D modèle", near_atd, DEAD),
                             ("3 + D modèle + SAM3", near_full, DEAD)):
        r = pm.evaluate(gt_codes, build(near, w, dead))
        rows.append((w, name, r["macro_f1"], r["spell_f1_mean"], r["n_pred_spells"]))
res = pd.DataFrame(rows, columns=["w", "variante", "macroF1", "spellF1", "possessions"])
for col in ("macroF1", "spellF1", "possessions"):
    print(f"\n{col} (vérité : {r['n_gt_spells']} possessions)" if col == "possessions" else f"\n{col}")
    print(res.pivot(index="w", columns="variante", values=col).round(4).to_string())

best = res[res["variante"].str.startswith("3")].assign(s=lambda d: d.macroF1 + d.spellF1).sort_values("s").iloc[-1]
W_BEST = int(best["w"])
FINAL = build(near_full, W_BEST, DEAD)                # prédiction finale, int8 (H/A/D)
print(f"\nw retenu = {W_BEST} | macroF1 {best['macroF1']:.4f} | spellF1 {best['spellF1']:.4f}")
np.savez("pipeline_possession.npz", dead=DEAD, ball=BALL, src=SRC, final=FINAL)

In [ ]:
idx = np.where(gt_codes != X)[0]
lo_f, hi_f = idx[0], idx[-1]
sl = slice(lo_f, hi_f + 1)

def n_possessions(seg):
    return int((seg[0] in (H, A)) + sum(seg[i] in (H, A) and seg[i] != seg[i - 1]
                                        for i in range(1, len(seg))))

# Séries : vérité + 3 variantes au w retenu
series = [("Vérité", gt_codes),
          (f"1 ATD seul (w={W_BEST})", build(near_atd, W_BEST)),
          (f"2 + D modèle (w={W_BEST})", build(near_atd, W_BEST, DEAD)),
          (f"3 + D + SAM3 (w={W_BEST})", FINAL)]

cmap = ListedColormap(["tab:blue", "tab:red", "lightgray", "white"])   # H, A, D, X
fig, axes = plt.subplots(len(series), 1, figsize=(12, 6), sharex=True)
for ax, (name, codes) in zip(axes, series):
    seg = codes[sl]
    ax.imshow(seg[None, :], aspect="auto", cmap=cmap, vmin=0, vmax=3, interpolation="nearest",
              extent=[lo_f / FPS, (hi_f + 1) / FPS, 0, 1])
    ax.set_yticks([])
    ax.set_ylabel(f"{name}\n{n_possessions(seg)} poss.", rotation=0, ha="right", va="center", fontsize=9)
axes[-1].set_xlabel("temps depuis le début de la vidéo (s)")
fig.suptitle("Bleu = H (Barça) · Rouge = A (Athletic) · Gris = D · Blanc = non noté", fontsize=10)
plt.tight_layout()
plt.show()